In [41]:
library(tidyverse)
library(oce)

In [73]:
ctd_ds <- read.csv("BCO-DMO/ctd.csv", na.strings="nd")

In [74]:
str(ctd_ds)

'data.frame':	288045 obs. of  21 variables:
 $ cruise_no : int  1 1 1 1 1 1 1 1 1 1 ...
 $ Cruise_ID1: chr  "93HG_001" "93HG_001" "93HG_001" "93HG_001" ...
 $ Cruise_ID2: chr  "CAR-001" "CAR-001" "CAR-001" "CAR-001" ...
 $ Year      : int  1995 1995 1995 1995 1995 1995 1995 1995 1995 1995 ...
 $ Month     : int  11 11 11 11 11 11 11 11 11 11 ...
 $ Day       : int  8 8 8 8 8 8 8 8 8 8 ...
 $ Date      : chr  "1995-11-08" "1995-11-08" "1995-11-08" "1995-11-08" ...
 $ Latitude  : num  10.5 10.5 10.5 10.5 10.5 10.5 10.5 10.5 10.5 10.5 ...
 $ Longitude : num  -64.7 -64.7 -64.7 -64.7 -64.7 ...
 $ press     : num  1.01 2.01 4.02 6.03 8.05 ...
 $ depth     : num  1 2 4 6 8 10 12 14 16 18 ...
 $ temp      : num  NA 27.5 27.5 27.5 27.5 ...
 $ sal       : num  36.6 36.6 36.6 36.6 36.6 ...
 $ potemp    : num  NA 27.5 27.5 27.5 27.5 ...
 $ sigma_t   : num  NA 23.8 23.8 23.8 23.8 ...
 $ sigma_0   : num  NA 23.8 23.8 23.8 23.8 ...
 $ O2_ml_L   : num  NA 3.96 3.98 3.98 3.99 ...
 $ beam_cp   : num  NA

In [75]:
ctd_ds$date <- paste(ctd_ds$Year,'-',ctd_ds$Month,'-',ctd_ds$Day, sep='')
ctd_ds$date <- as.Date(ctd_ds$date, format="%Y-%m-%d")

In [45]:
# import interpolation functions
source('interpolateData.r')

In [46]:
ctd_temp_int = interpolateDF(prepdataframe(ctd_ds, "temp"))

In [47]:
iso21_depth <- ctd_temp_int %>%
    group_by(date) %>%
    filter(depth > 6) %>%
    mutate(iso21 = value_int < 21) %>% # create new column that gives "True" for values at MLD
    filter(iso21 == T) %>% # only take "True" values 
    slice(1) # takes the first occurrence

In [48]:
iso21_df <- iso21_depth %>%
    rename(Isotherm_21 = depth) %>%
    select(date, Isotherm_21)

In [49]:
# MLD from sigma_t

In [50]:
ctd_sigma_t_int = interpolateDF(prepdataframe(ctd_ds, "sigma_t"))

In [51]:
ctd_sigma_t_diff <- ctd_sigma_t_int %>%
    group_by(date) %>%
    do(data.frame( sigma_t = .$value_int, sigma_t_diff = c(NA,diff(.$value_int)), depth = .$depth))

head(ctd_sigma_t_diff)

date,sigma_t,sigma_t_diff,depth
<date>,<dbl>,<dbl>,<dbl>
1995-11-08,23.76400,NA,0
1995-11-08,23.76400,0.000000000,1
1995-11-08,23.76400,0.000000000,2
1995-11-08,23.76950,0.005500000,3
1995-11-08,23.77500,0.005500000,4
1995-11-08,23.77895,0.003947205,5


In [52]:
# Criterion 1
mld_depth <- ctd_sigma_t_diff %>%
    group_by(date) %>%
    filter(depth > 9) %>%
    mutate(mld = sigma_t_diff >= 0.125 | sigma_t_diff <= -0.125) %>% # create new column that gives "True" for values at MLD
    filter(mld == T) %>% # only take "True" values 
    slice(1) # takes the first occurrence

In [53]:
#Criterion 2 -- better apparently (i think from some paper or email!)
mld_depth_2 <- ctd_sigma_t_diff %>%
    group_by(date) %>%
    filter(depth > 9) %>%
    mutate(mld = sigma_t >= sigma_t[1]+0.2 | sigma_t <= sigma_t[1]-0.2) %>% # create new column that gives "True" for values at MLD
    filter(mld == T) %>% # only take "True" values 
    slice(1) # takes the first occurrence

In [54]:
mld_df <- mld_depth_2 %>%
    rename(MLD = depth) %>%
    select(date, MLD)

In [55]:
# Calculate SST from Temperature at surface
sst_10m <- ctd_temp_int %>%
  group_by(date) %>%
  filter(depth <= 10) %>%
  summarize(sst = mean(value_int))

In [56]:
# Combine Isotherm and MLD data frames
CTD_combined_data <- list(iso21_df, mld_df, sst_10m) %>% 
  reduce(left_join, by = "date") %>% as.data.frame()

#write.csv(CTD_combined_data, "../DATA/January/CTD_Isotherm21_MLD.csv")

In [57]:
write.csv(CTD_combined_data, "processed/CTD_Isotherm21_MLD.csv")

In [58]:
saveRDS(CTD_combined_data, "processed/CTD_Isotherm21_MLD.rds")

# calculate upwelling index

In [39]:
UpwellingIndex <- ctd_temp_int %>%
    group_by(date) %>%
    filter(depth == 50) %>%
    mutate(ui = case_when(
                         value_int<=20.0 ~ "strong",
                         value_int<=21.0 ~ "moderate",
                         value_int<=22.0 ~ "weak",
                         value_int>22.0 ~ "relaxed"),
          upwelling = case_when(
                         value_int<=22.0 ~ "upwelling",
                         value_int>22.0 ~ "relaxed"),
          ) %>% select(date, ui, upwelling)

head(UpwellingIndex)

date,ui,upwelling
<date>,<chr>,<chr>
1995-11-08,relaxed,relaxed
1995-12-14,relaxed,relaxed
1996-01-13,relaxed,relaxed
1996-02-14,relaxed,relaxed
1996-03-13,moderate,upwelling
1996-04-16,moderate,upwelling


In [40]:
saveRDS(UpwellingIndex, "processed/CTD_UpwellingIndex.rds")